# Build the Visium HD Spatial Object

**Goal.** Prepare a QC-filtered spatial object (`adata_sp`) from the public Visium HD data
(GSE288758; bioRxiv 10.1101/2025.02.10.637543; *Mol Cancer* 2025, doi:10.1186/s12943-025-02493-6)

**Data.** Two slides of StarDist-segmented cells with the authors' cell-type labels:
slide 1 = early resistance (ER) and standard response (SR), slide 2 = two long-term responders
(LR1, LR2); each patient has pre- and post-treatment sections (`sample`, e.g. `ER_pre`).

**Steps.** Load slides → harmonise labels (`cell_type_2`, all `tc_*` tumour clusters → `tumour_cell`)
→ merge with one spatial library per slide → QC filter → add `response_group` and raw counts → save
→ H&E overlays per cell type.

**Output.** `adata1_adata2_preprocess_join.h5ad` (raw counts in `X` and `layers['raw_counts']`).

In [ ]:
# !pip install --quiet scanpy squidpy
# %cd <project>/spatial_visiumHD

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq

## Config

In [ ]:
slides = {      # slide name -> processed h5ad from the authors
    "slide1": "slide1/slide1_stardist_integrated.h5ad",
    "slide2": "slide2/slide2_stardist_integrated.h5ad",
}

output_path = "adata1_adata2_preprocess_join.h5ad"
figures_output_dir = "figures"
tables_output_dir = "tables"

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)

## 1. Load slides and harmonise labels

Each slide's spatial library (H&E image + scale factors) is renamed to the slide name, so both
images survive the merge and squidpy can plot each slide by `library_id`.

In [ ]:
# --- labels ---
source_label_col = "cell_type"          # authors' annotation (tumour clusters named tc_*)
label_col = "cell_type_2"               # harmonised label
tumour_prefix, tumour_label = "tc_", "tumour_cell"
label_map = {
    "alpha beta T cell": "T cell", "gamma delta T cell": "T cell",
    "Activated B cell": "B cell", "Naive B cell": "B cell",
    "Plasmablast IgG1": "Plasmablast", "Plasmablast IgA": "Plasmablast",
    "Plasmablast IgG3": "Plasmablast", "Plasmablast IgM": "Plasmablast",
    "Goblet/Ciliated cell": "Goblet Ciliated cell",
}
sample_col = "sample"                   # section, e.g. ER_pre; response group = text before "_"

def load_slide(name, path):
    a = sc.read_h5ad(path)
    libs = list(a.uns["spatial"])
    assert len(libs) == 1, f"{name}: expected one spatial library, found {libs}"
    a.uns["spatial"] = {name: a.uns["spatial"][libs[0]]}
    a.obs["slide"] = name
    a.obs_names = [f"{name}_{x}" for x in a.obs_names]

    labels = a.obs[source_label_col].astype(str)
    labels[labels.str.startswith(tumour_prefix)] = tumour_label
    a.obs[label_col] = labels.replace(label_map)
    print(f"{name}: {a.n_obs} cells, {a.n_vars} genes, samples {sorted(a.obs[sample_col].unique())}")
    return a


parts = {name: load_slide(name, path) for name, path in slides.items()}
pd.concat({n: a.obs[label_col].value_counts() for n, a in parts.items()}, axis=1).fillna(0).astype(int)

In [ ]:
for name, a in parts.items():
    fig, ax = plt.subplots(figsize=(8, 8))
    sq.pl.spatial_scatter(a, library_id=name, img=True, color=None, size=0, ax=ax)
    plt.savefig(f"{figures_output_dir}/{name}_HnE.pdf", bbox_inches="tight")
    plt.show()

## 2. Merge slides

In [ ]:
adata_sp = sc.concat(list(parts.values()), join="outer", merge="same")
adata_sp.uns["spatial"] = {name: a.uns["spatial"][name] for name, a in parts.items()}
assert adata_sp.obs_names.is_unique
del parts
adata_sp

## 3. QC filter

Distributions are shown **before** filtering with the thresholds marked; the table reports
cells removed per section.

In [ ]:
# --- QC (per segmented cell) ---
min_genes, min_counts, max_pct_mt = 25, 25, 5
mt_prefix = "MT-"

adata_sp.var["mt"] = adata_sp.var_names.str.upper().str.startswith(mt_prefix)
sc.pp.calculate_qc_metrics(adata_sp, qc_vars=["mt"], inplace=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, thr) in zip(axes, [("n_genes_by_counts", min_genes), ("total_counts", min_counts),
                                 ("pct_counts_mt", max_pct_mt)]):
    sns.histplot(adata_sp.obs[col], bins=60, ax=ax, log_scale=(col != "pct_counts_mt", False))
    ax.axvline(thr, ls="--", c="red")
plt.tight_layout()
plt.savefig(f"{figures_output_dir}/qc_histograms_thresholds.pdf", bbox_inches="tight")
plt.show()

keep = ((adata_sp.obs["n_genes_by_counts"] >= min_genes) & (adata_sp.obs["total_counts"] >= min_counts)
        & (adata_sp.obs["pct_counts_mt"] < max_pct_mt))
qc_table = pd.DataFrame({"before": adata_sp.obs[sample_col].value_counts(),
                         "after": adata_sp.obs.loc[keep, sample_col].value_counts()})
qc_table["pct_removed"] = (100 * (1 - qc_table["after"] / qc_table["before"])).round(1)
qc_table.to_csv(f"{tables_output_dir}/qc_cells_per_sample.csv")
adata_sp = adata_sp[keep].copy()
qc_table

## 4. Metadata, raw counts and save

In [ ]:
adata_sp.obs["response_group"] = adata_sp.obs[sample_col].astype(str).str.split("_").str[0].astype("category")
vals = adata_sp.X[:1000].data
assert np.allclose(vals, np.round(vals)), "X must contain raw counts"
adata_sp.layers["raw_counts"] = adata_sp.X.copy()

print(pd.crosstab(adata_sp.obs[label_col], adata_sp.obs["response_group"], margins=True))
adata_sp.write(output_path)
adata_sp

## 5. H&E overlays per cell type

One panel per cell type and slide (cell type coloured on the H&E image).

In [ ]:
# --- H&E overlays ---
celltype_colors = {
    "tumour_cell": "#FFD700", "Plasmablast": "#FF0000", "Macrophage": "#0000FF", "Endothelial cell": "#00FFFF",
    "Fibroblast": "#FF00FF", "Smooth muscle cell": "#00FF00", "T cell": "#FFA500", "B cell": "#800080",
    "Goblet Ciliated cell": "#00CED1", "Alveolar cell type 2": "#7FFF00", "Follicular dendritic cell": "#DC143C",
    "Basal cell": "#FF1493", "Granulocyte": "#8B0000", "Club cell": "#1E90FF", "Erythrocyte": "#B22222",
}
overlay_cell_types = list(celltype_colors)   # one H&E overlay per cell type and slide
point_size = 15

def overlay(adata, slide, celltype, color):
    sub = adata[(adata.obs["slide"] == slide) & (adata.obs[label_col] == celltype)].copy()
    if sub.n_obs == 0:
        return
    sub.obs[label_col] = pd.Categorical(sub.obs[label_col])
    sub.uns[f"{label_col}_colors"] = [color]
    fig, ax = plt.subplots(figsize=(8, 8))
    sq.pl.spatial_scatter(sub, library_id=slide, img=True, color=label_col, size=point_size, ax=ax)
    ax.set_title(f"{celltype} ({sub.n_obs} cells)", fontsize=14)
    plt.savefig(f"{figures_output_dir}/{slide}_{celltype.replace(' ', '_')}_on_HnE.pdf", bbox_inches="tight")
    plt.show()


for celltype in overlay_cell_types:
    for slide in slides:
        overlay(adata_sp, slide, celltype, celltype_colors[celltype])